# RAG Pipeline with Hugging Face Docs, LangChain, FAISS, and FLAN-T5

This notebook builds a small Retrieval-Augmented Generation (RAG) pipeline over the `m-ric/huggingface_doc` dataset, following the task list:

1. Setup and imports
2. Load the dataset
3. Convert rows into LangChain `Document` objects
4. Chunk the documents
5. Build the vector store and retriever
6. Retrieval sanity check
7. RAG question answering

## 1. Setup and imports

Install dependencies, then import the dataset loader, Hugging Face pipeline utilities, and the LangChain pieces needed for chunking, embedding, retrieval, and QA.

In [ ]:
# Run this once. Restart the kernel afterward if any packages were newly installed.
!pip install -q -U datasets langchain langchain-community sentence-transformers faiss-cpu transformers accelerate


In [ ]:
from datasets import load_dataset

from transformers import pipeline

from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA


## 2. Load the dataset

Load `m-ric/huggingface_doc`, inspect its columns, and look at one raw example row.

In [ ]:
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print("Number of rows:", len(dataset))
print("Columns:", dataset.column_names)


In [ ]:
# Look at one example row to see the raw structure (text + source, typically)
dataset[0]


## 3. Convert rows into LangChain `Document` objects

Each `Document` stores its main text in `page_content` and a meaningful `source` inside `metadata`, so retrieved chunks can later be traced back to where they came from.

In [ ]:
docs = []
for row in dataset:
    text = row["text"]
    source = row.get("source", "unknown")
    docs.append(Document(page_content=text, metadata={"source": source}))

print(f"Total documents: {len(docs)}")
print("Example metadata:", docs[0].metadata)
print("Example content (first 300 chars):\n", docs[0].page_content[:300])


> **Note:** The full dataset has several thousand rows. For faster experimentation in this notebook, the chunking and indexing steps below use a subset (`docs[:200]`). Increase or remove the slice once you're happy with the pipeline and have time/compute to spare.

## 4. Chunk the documents

Split the documents with `RecursiveCharacterTextSplitter`, trying two different `(chunk_size, chunk_overlap)` settings, and compare the resulting chunks.

In [ ]:
doc_subset = docs[:200]  # adjust/remove the slice as needed

# Strategy A: smaller, more granular chunks
splitter_a = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks_a = splitter_a.split_documents(doc_subset)

# Strategy B: larger chunks with more overlap
splitter_b = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks_b = splitter_b.split_documents(doc_subset)

print(f"Strategy A (chunk_size=500, overlap=50): {len(chunks_a)} chunks")
print(f"Strategy B (chunk_size=1000, overlap=100): {len(chunks_b)} chunks")


In [ ]:
print("--- Sample chunk from Strategy A ---")
print(chunks_a[0].metadata)
print(chunks_a[0].page_content[:300])

print("\n--- Sample chunk from Strategy B ---")
print(chunks_b[0].metadata)
print(chunks_b[0].page_content[:300])


**Observation prompt:** Compare chunk counts and content boundaries between Strategy A and B. Smaller chunks (A) tend to be more focused but may cut sentences/context; larger chunks (B) preserve more context per chunk but can dilute relevance when a retrieved chunk only partially matches the question. Pick one strategy to carry forward (this notebook uses `chunks_b` in step 5, but you can switch to `chunks_a`).

## 5. Build the vector store and retriever

Embed the chunks with `sentence-transformers/all-MiniLM-L6-v2`, build a FAISS index, and create retrievers with `k=4` (the default to use going forward), plus `k=2` and `k=6` for comparison.

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Choose which chunking strategy to index (swap in chunks_a to compare)
chunks = chunks_b

vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever_k2 = vectorstore.as_retriever(search_kwargs={"k": 2})
retriever_k4 = vectorstore.as_retriever(search_kwargs={"k": 4})
retriever_k6 = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Vector store built with", vectorstore.index.ntotal, "vectors.")


## 6. Retrieval sanity check (required)

Before generating any answers, manually inspect retrieved chunks for a test question across the three `k` values. If chunks look irrelevant, go back and adjust chunking parameters and/or `k`.

In [ ]:
test_question = "How do I load a dataset with the datasets library?"

for label, retriever in [("k=2", retriever_k2), ("k=4", retriever_k4), ("k=6", retriever_k6)]:
    print(f"\n=== Retrieval with {label} ===")
    results = retriever.invoke(test_question)
    for i, r in enumerate(results):
        print(f"\n--- Chunk {i + 1} | source: {r.metadata.get('source')} ---")
        print(r.page_content[:300])


**Checklist before moving on:**
- Do the retrieved chunks actually mention loading datasets / the `datasets` library?
- Does increasing `k` from 2 → 6 add genuinely useful chunks, or mostly noise?
- Do the `source` values look meaningful (not all `unknown`)?

If the answer to any of these is "no," go back to step 4 and try a different `chunk_size`/`chunk_overlap`, or rebuild the index in step 5 with a larger `doc_subset`, before continuing.

## 7. RAG question answering

Set up a small local LLM (`google/flan-t5-small`), wrap it in a `RetrievalQA` chain with the `k=4` retriever, ask several questions, and print both the answer and the sources of the retrieved chunks.

In [ ]:
hf_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=256,
)
llm = HuggingFacePipeline(pipeline=hf_pipe)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever_k4,
    return_source_documents=True,
)


In [ ]:
questions = [
    "How do I load a dataset with the datasets library?",
    "What is a Hugging Face pipeline?",
    "How do I push a model to the Hugging Face Hub?",
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print("Sources used:")
    for doc in result["source_documents"]:
        print(f"  - {doc.metadata.get('source')}")
    print("-" * 80)


**Evaluating the answers:** `flan-t5-small` is a small, weak model, so answers may be terse or occasionally miss the point even with good context. Focus less on eloquence and more on whether:
- The printed **sources** are actually relevant to the question (this tells you retrieval is working).
- The **answer** at least paraphrases something present in the retrieved chunks, rather than sounding like unrelated generic text (this tells you the model is grounding on the dataset rather than guessing).